# Fase 3 — Sistema completo vs mono (máximo estrés espacial)

Fase final: **máxima diversidad acústica** y **mínima diversidad de procesadores**
(sólo el combo ganador de las fases previas; a lo sumo 1-2 combos para evaluar costo
computacional). Se compara el **sistema completo** (NM-MVDR + post-filtro elegido en
la Fase 2, **sin WPE**) contra el **DTLN monocanal**.

Un **único grid combinado** donde iSIR × RT60 × nº-interferentes × locutores varían
en conjunto → **boxplots marginales** (la dispersión de cada caja = incertidumbre
real del envelope operativo) y **diferencia pareada** BF − mono.

El sesgo espacial del eje "nº interferentes" se controla con un esquema **nested**
(conteo k+1 = conteo k + una fuente) en 2 rotaciones espejo, lados alternados y
audios rotados → aísla el efecto del *número* del de la *posición*.

**Antes de correr:** poné `SYSTEM_SMOOTH` = el `smooth*` elegido en la Fase 2.

**Correr en orden:** Setup → Config+Run → Boxplots marginales → Diferencia pareada.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive

# CONSTRUIR (git+ para las libs de GitHub).
BUILD = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "git+https://github.com/fgnt/pb_bss.git",
    "git+https://github.com/LCAV/pyroomacoustics.git",
    "git+https://github.com/fgnt/nara_wpe.git",
    "git+https://github.com/fakufaku/fast_bss_eval.git",
]
# INSTALAR desde cache: NOMBRES (no git+, si no pip vuelve a clonar).
INSTALL = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "pb_bss", "pyroomacoustics", "nara_wpe", "fast_bss_eval",
]

# Reconstruye el cache SOLO si la lista de paquetes cambio (manifest) -> se
# autocura si agrego/saco un paquete, sin tener que borrar el cache a mano.
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
need_build = (not os.path.isfile(manifest)) or open(manifest).read() != key

if need_build:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    !pip wheel --wheel-dir=$WHL {" ".join(BUILD)}
    with open(manifest, "w") as fh:
        fh.write(key)
    print("[*] Cache actualizado en", WHL)

!pip install --no-index --find-links=$WHL {" ".join(INSTALL)}
print("[*] Paquetes instalados desde el cache de Drive.")
# Si Colab actualiza Python y falla un import:  !rm -rf $WHL  (se reconstruye solo)

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config + Ejecución (Fase 3)

In [ ]:
import sys, os, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import NM_MVDR, NM_MVDR_BAN_PF
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK.")
else:
    print("[!] Sin DTLN interpreters.")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION      = 10
SYSTEM_SMOOTH = 0.33              # <<< PLACEHOLDER: smooth* elegido en la Fase 2
ISIR_LIST     = [-6, -3, 0, 3, 6, 9, 12]   # denso, paso 3 dB
RT60_LIST     = [0.160, 0.360, 0.610]      # partí por RT acá si no entra en la sesión
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
    "p011_emo_anger_sentences.wav",
]]
# Interferentes NO vocales (DTLN es supresión, no separación). idx 0..5
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav", "hairdryer_07_SH_MKH800.wav", "drill_07_RHODE_NT1.wav",
    "ruido_rosa_16k.wav", "vacuum_02_RHODE_NT1.wav", "flute_music.wav",
]]

# --- Esquema NESTED para el eje nº-interferentes (máximo estrés espacial) ---
# Rot A: base 30, agrega -45, 60, -90 (lados alternados, hacia endfire).
# Rot B: espejo (-30,45,-60,90) con audios rotados. audio_idx = 3er elemento.
A1=[(30,1.0,0)]
A2=[(30,1.0,0),(-45,1.0,1)]
A3=[(30,1.0,0),(-45,1.0,1),(60,1.0,2)]
A4=[(30,1.0,0),(-45,1.0,1),(60,1.0,2),(-90,1.0,3)]
B1=[(-30,1.0,4)]
B2=[(-30,1.0,4),(45,1.0,5)]
B3=[(-30,1.0,4),(45,1.0,5),(-60,1.0,0)]
B4=[(-30,1.0,4),(45,1.0,5),(-60,1.0,0),(90,1.0,1)]
INTERF_CONFIGS=[A1,A2,A3,A4,B1,B2,B3,B4]   # 8 escenas, 2 por conteo (1..4)

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.008,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}

param_grid = {
    'rt60':RT60_LIST, 'target_angle':[0], 'target_dist':[1.0],
    'source_path':TARGETS, 'interf_configs':INTERF_CONFIGS,
    'isir_db':ISIR_LIST, 'use_wpe':[False],
    'mismatch_gain':[0], 'mismatch_phase':[0],
    'error_angle_deg':[0.0], 'error_distance_m':[0.0],
}

# --- PROCESADORES (Fase 3: mínima diversidad = combo ganador) ---
# PLACEHOLDER: reemplazá "Sistema" por el combo ganador de las fases previas. Podés
# dejar 1-2 combos para comparar costo computacional (exec_time_s en el CSV).
processors_dict = {
    "NM-MVDR": NM_MVDR(min_loading=1e-6, alpha=0.99),                          # base
    "Sistema": NM_MVDR_BAN_PF(min_loading=1e-6, alpha=0.99, smooth=SYSTEM_SMOOTH),
}

n = len(RT60_LIST)*len(TARGETS)*len(INTERF_CONFIGS)*len(ISIR_LIST)
print("="*60)
print(f"FASE 3 | celdas={n} x {len(processors_dict)} proc | smooth*={SYSTEM_SMOOTH}")
print("="*60)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp  = f"/content/results_temp/F3_sys_vs_mono_{RUN_TAG}"
drive = f"/content/drive/MyDrive/Tesis_Beamformers/results/F3_sys_vs_mono_{RUN_TAG}"
os.makedirs(temp, exist_ok=True); os.makedirs(drive, exist_ok=True)

df_F3 = run_mird_grid_search(grid_params=param_grid, dataset_provider=provider,
    processors=processors_dict, scene_base_config=base_config, output_dir=temp,
    interpreter_1=interpreter_1, interpreter_2=interpreter_2,
    save_catalog=False, apply_dtln_post=False)
shutil.copytree(temp, drive, dirs_exist_ok=True)
print(f"[EXITO] Fase 3 -> {drive}")

## Boxplots marginales — sistema vs NM-MVDR vs mono

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(os.path.join(drive, "mird_benchmark_metrics.csv"))
SYSTEM  = "Sistema"
METRICS = ["SDR", "SIR", "STOI", "PESQ"]
PAL = {"NM-MVDR":"tab:orange", "Sistema":"tab:blue", "DTLN-mono":"tab:gray"}

# long-form: métricas ABSOLUTAS (proc_* para los BF, dtln_alone_* para el mono)
def tidy(df):
    rows=[]
    for method,proc in [("NM-MVDR","NM-MVDR"),("Sistema",SYSTEM)]:
        for _,r in df[df.processor==proc].iterrows():
            for M in METRICS:
                rows.append({"method":method,"metric":M,"value":r.get(f"proc_{M}_early",np.nan),
                             "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
    for _,r in df[df.processor=="NM-MVDR"].iterrows():   # mono: 1 vez por celda
        for M in METRICS:
            rows.append({"method":"DTLN-mono","metric":M,"value":r.get(f"dtln_alone_{M}_early",np.nan),
                         "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
    return pd.DataFrame(rows).dropna(subset=["value"])

T = tidy(df)
order = ["NM-MVDR","Sistema","DTLN-mono"]

def boxfig(xvar, xlabel, title, fname):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, M in zip(axes.ravel(), METRICS):
        d = T[T.metric==M]
        sns.boxplot(data=d, x=xvar, y="value", hue="method", hue_order=order,
                    palette=PAL, ax=ax, fliersize=1, linewidth=0.8)
        ax.set_xlabel(xlabel); ax.set_ylabel(M); ax.grid(alpha=0.3, axis="y")
        if ax.get_legend(): ax.get_legend().remove()
    axes.ravel()[0].legend(fontsize=8, loc="best")
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(drive, fname), dpi=140, bbox_inches="tight"); plt.show()

boxfig("iSIR", "iSIR [dB]",        "Fase 3 — sistema vs mono vs iSIR",          "F3_vs_isir.png")
boxfig("RT",   "RT60 [ms]",        "Fase 3 — vs RT60",                          "F3_vs_rt.png")
boxfig("N",    "nº interferentes", "Fase 3 — vs nº de interferentes",           "F3_vs_count.png")

## Diferencia pareada (BF − mono) — dónde gana cada uno

In [ ]:
# proc_* y dtln_alone_* viven en la MISMA fila -> diferencia pareada por celda.
def tidy_diff(df):
    rows=[]
    for method,proc in [("NM-MVDR−mono","NM-MVDR"),("Sistema−mono",SYSTEM)]:
        for _,r in df[df.processor==proc].iterrows():
            for M in METRICS:
                d = r.get(f"proc_{M}_early",np.nan) - r.get(f"dtln_alone_{M}_early",np.nan)
                rows.append({"cmp":method,"metric":M,"diff":d,
                             "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
    return pd.DataFrame(rows).dropna(subset=["diff"])

D = tidy_diff(df)
PALD = {"NM-MVDR−mono":"tab:orange", "Sistema−mono":"tab:blue"}

def diffig(xvar, xlabel, title, fname):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, M in zip(axes.ravel(), METRICS):
        d = D[D.metric==M]
        sns.boxplot(data=d, x=xvar, y="diff", hue="cmp",
                    hue_order=["NM-MVDR−mono","Sistema−mono"], palette=PALD,
                    ax=ax, fliersize=1, linewidth=0.8)
        ax.axhline(0, color="k", lw=0.8, ls="--")   # arriba de 0 = el BF le gana al mono
        ax.set_xlabel(xlabel); ax.set_ylabel(f"Δ {M} (BF − mono)"); ax.grid(alpha=0.3, axis="y")
        if ax.get_legend(): ax.get_legend().remove()
    axes.ravel()[0].legend(fontsize=8, loc="best")
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(drive, fname), dpi=140, bbox_inches="tight"); plt.show()

diffig("iSIR", "iSIR [dB]",        "Fase 3 — ventaja sobre mono vs iSIR",        "F3_diff_isir.png")
diffig("RT",   "RT60 [ms]",        "Fase 3 — ventaja sobre mono vs RT60",        "F3_diff_rt.png")
diffig("N",    "nº interferentes", "Fase 3 — ventaja sobre mono vs nº interf",   "F3_diff_count.png")